# Baselines on Blotto: R-PPO, DR-RPPO, VariBAD, PEARL, and MMD
Every baseline acts on the same observation interface as our defender, its latest control reading and its previous allocation $[y_k, a^1_{k-1}/B]$, through a recurrent policy, and every baseline trains for the same number of environment steps as our method (`BUDGET_STEPS`).

- **R-PPO**: recurrent PPO against the trained attacker.
- **DR-RPPO**: recurrent PPO against an attacker drawn per episode from a randomized pool of admissible attackers.
- **VariBAD**: Bayes-adaptive meta-RL, conditioning the policy on an inferred task belief (mean and standard deviation of a recurrent variational encoder trained to predict the next observation).
- **PEARL**: latent-context meta-RL acting by posterior sampling, resampling the latent every `PEARL_RESAMPLE_K` steps from a product-of-Gaussians posterior over the episode's transitions.
- **MMD**: magnetic mirror descent in self-play, regularizing each player toward a slowly moving magnet copy of itself; it respects the game but never represents the context.

`RecurrentAC`, `VariBADEncoder`, `PEARLEncoder`, and `sample_config` are defined in `Initialize.ipynb`. Checkpoints are saved in `compared/`, each as a dictionary holding the network weights, the configuration, and the per-update reward per step.

In [ ]:
%run Initialize.ipynb

## Shared settings

In [ ]:
from copy import deepcopy
from torch.distributions import kl_divergence

# Matched interaction budget: main.ipynb trains our method for 10,000 episodes x 50 steps
BUDGET_STEPS     = 10000 * max_steps
TOTAL_EPISODES   = BUDGET_STEPS // max_steps     # 10,000 episodes for every baseline
EPISODES_PER_UPD = 1
GAMMA            = 0.95
GAE_LAMBDA       = 0.95
PPO_EPOCHS       = 4
CLIP_EPS         = 0.2
ENT_COEF         = 0.01
VF_COEF          = 0.5
LR               = 3e-4
VARIBAD_KL_W     = 0.05
PEARL_KL_W       = 0.1
PEARL_RESAMPLE_K = 10
MMD_COEF         = 0.05
MAGNET_TAU       = 0.1
CKPT_EVERY       = 200
SAVE_DIR         = "compared"
os.makedirs(SAVE_DIR, exist_ok=True)
BUDGET = {}                                      # environment steps used by each baseline

Player2_fixed = Load(Player2Actor().to(device), "player2.pth")

## Attacker pool (every member respects the reachability mask)

In [ ]:
def vec_to_idx(vec):
    return action_to_idx[tuple(int(v) for v in vec)]

class OpponentTrainedP2:
    """The trained attacker; it needs its private state m2."""
    needs_m2 = True
    def reset(self, Theta, Weights, Reachable, legal):
        self.tt = torch.tensor(np.concatenate([Weights, Reachable]),
                               dtype=torch.float32, device=device)
    def act(self, x, m2):
        with torch.no_grad():
            p = Player2_fixed(torch.tensor(x, dtype=torch.float32, device=device).unsqueeze(0),
                              torch.tensor(m2, dtype=torch.float32, device=device).unsqueeze(0),
                              self.tt.unsqueeze(0)).squeeze(0).cpu().numpy()
        p = np.clip(p, 0, None); p = p / p.sum()
        return int(np.random.choice(n_u, p=p))

class OpponentRandom:
    """Uniform over legal attacker actions."""
    needs_m2 = False
    def reset(self, Theta, Weights, Reachable, legal):
        self.legal = legal
    def act(self, x, m2):
        return int(np.random.choice(self.legal))

class OpponentOneTown:
    """All budget on one random reachable town, redrawn per episode."""
    needs_m2 = False
    def reset(self, Theta, Weights, Reachable, legal):
        town = int(np.random.choice(np.where(Reachable > 0)[0]))
        vec = np.zeros(N, dtype=int); vec[town] = B
        self.idx = vec_to_idx(vec)
    def act(self, x, m2):
        return self.idx

class OpponentGreedy:
    """All budget on the highest-weight reachable town."""
    needs_m2 = False
    def reset(self, Theta, Weights, Reachable, legal):
        reach = np.where(Reachable > 0)[0]
        town = int(reach[np.argmax(Weights[reach])])
        vec = np.zeros(N, dtype=int); vec[town] = B
        self.idx = vec_to_idx(vec)
    def act(self, x, m2):
        return self.idx

class OpponentProportional:
    """Each troop placed i.i.d. over reachable towns proportional to weights."""
    needs_m2 = False
    def reset(self, Theta, Weights, Reachable, legal):
        w = Weights * Reachable
        self.p = w / w.sum() if w.sum() > 0 else Reachable / Reachable.sum()
    def act(self, x, m2):
        return vec_to_idx(np.random.multinomial(B, self.p))

POOL_FACTORIES = [OpponentRandom, OpponentOneTown, OpponentGreedy,
                  OpponentProportional, OpponentTrainedP2]

## Rollouts and PPO

In [ ]:
def gae(rews, vals):
    T = len(rews)
    adv = np.zeros(T, dtype=np.float32); last = 0.0
    for t in reversed(range(T)):
        nv = vals[t + 1] if t + 1 < T else 0.0    # horizon ends at max_steps
        delta = rews[t] + GAMMA * nv - vals[t]
        last = delta + GAMMA * GAE_LAMBDA * last
        adv[t] = last
    return adv, adv + np.asarray(vals, dtype=np.float32)

def rollout_episode(policy, opponent, hook=None):
    Theta, Weights, Reachable, x, legal = sample_config()
    opponent.reset(Theta, Weights, Reachable, legal)
    m2 = np.full(N * 2, 0.5)
    h = None
    y_cur = np.zeros(N); a_prev = np.zeros(N)
    if hook is not None:
        hook.reset()
    obs_l, ex_l, a_l, lp_l, v_l, r_l, y_l, av_l = [], [], [], [], [], [], [], []
    for t in range(max_steps):
        obs = np.concatenate([y_cur, a_prev]).astype(np.float32)
        obs_t = torch.tensor(obs, device=device).view(1, 1, 2 * N)
        ex_np = hook.extra(obs) if hook is not None else None
        ex_t = None if ex_np is None else torch.tensor(
            np.asarray(ex_np, dtype=np.float32), device=device).view(1, 1, -1)
        with torch.no_grad():
            logits, val, h = policy(obs_t, ex_t, h)
        dist = Categorical(logits=logits.view(-1))
        a = int(dist.sample())
        lp = float(dist.log_prob(torch.tensor(a, device=device)))
        u = opponent.act(x, m2)
        a_vec = actions[a]; u_vec = actions[u]
        nx = next_state(x, a_vec, u_vec, Theta)
        r = float(reward(nx, Theta))
        if opponent.needs_m2:
            m2 = m2_update(m2, x, u_vec, nx)
        y = Observation(nx)
        if hook is not None:
            hook.observe(y_cur, a_vec / B, y)
        obs_l.append(obs); a_l.append(a); lp_l.append(lp); v_l.append(val.item())
        r_l.append(r); y_l.append(y.astype(np.float32)); av_l.append((a_vec / B).astype(np.float32))
        if ex_np is not None:
            ex_l.append(np.asarray(ex_np, dtype=np.float32))
        x = nx; y_cur = y; a_prev = a_vec / B
    adv, ret = gae(np.array(r_l, dtype=np.float32), np.array(v_l, dtype=np.float32))
    ep = {'obs': np.stack(obs_l), 'acts': np.array(a_l), 'logp': np.array(lp_l, dtype=np.float32),
          'adv': adv, 'ret': ret, 'rews': np.array(r_l, dtype=np.float32),
          'ys': np.stack(y_l), 'avecs': np.stack(av_l)}
    if ex_l:
        ep['extras'] = np.stack(ex_l)
    return ep

def stack_batch(eps, keys):
    out = []
    for k in keys:
        dt = torch.long if k == 'acts' else torch.float32
        out.append(torch.tensor(np.stack([e[k] for e in eps]), dtype=dt, device=device))
    return out

def ppo_update(policy, opt, eps, extras=None, extra_params=None):
    obs, acts, oldlp, adv, ret = stack_batch(eps, ['obs', 'acts', 'logp', 'adv', 'ret'])
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)
    params = list(policy.parameters()) + (list(extra_params) if extra_params else [])
    for _ in range(PPO_EPOCHS):
        logits, values, _ = policy(obs, extras)
        dist = Categorical(logits=logits)
        lp = dist.log_prob(acts)
        ratio = (lp - oldlp).exp()
        pg = -torch.min(ratio * adv,
                        torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * adv).mean()
        vl = F.mse_loss(values, ret)
        ent = dist.entropy().mean()
        loss = pg + VF_COEF * vl - ENT_COEF * ent
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        opt.step()
    return float(pg.item()), float(vl.item())

In [ ]:
def train_recurrent_ppo(save_name, opponent_sampler, seed):
    np.random.seed(seed); torch.manual_seed(seed); random.seed(seed)
    policy = RecurrentAC(extra_dim=0).to(device)
    opt = optim.AdamW(policy.parameters(), lr=LR)
    n_upd = TOTAL_EPISODES // EPISODES_PER_UPD
    ret_hist = []
    for upd in tqdm(range(n_upd), desc=save_name):
        eps = [rollout_episode(policy, opponent_sampler())
               for _ in range(EPISODES_PER_UPD)]
        ppo_update(policy, opt, eps)
        ret_hist.append(float(np.mean([e['rews'].mean() for e in eps])))
        if (upd + 1) % CKPT_EVERY == 0 or upd == n_upd - 1:
            torch.save({'policy': policy.state_dict(),
                        'config': {'RNN_H': RNN_H, 'extra_dim': 0},
                        'reward_per_step': ret_hist},
                       os.path.join(SAVE_DIR, save_name + '.pth'))
        if (upd + 1) % 25 == 0:
            print(f"{save_name} upd {upd+1}/{n_upd}  reward/step (last 25 upd): "
                  f"{np.mean(ret_hist[-25:]):.3f}")
    BUDGET[save_name] = n_upd * EPISODES_PER_UPD * max_steps
    return policy, ret_hist

## R-PPO: recurrent PPO against the trained attacker

In [ ]:
RPPO_policy, RPPO_hist = train_recurrent_ppo('RPPO', lambda: OpponentTrainedP2(), seed=101)

## DR-RPPO: recurrent PPO against an attacker drawn per episode from the pool

In [ ]:
DR_policy, DR_hist = train_recurrent_ppo('DR_RPPO', lambda: random.choice(POOL_FACTORIES)(), seed=202)

## VariBAD: policy conditioned on a variational task belief

In [ ]:
class VariBADDecoder(nn.Module):
    """Predicts y_{t+1} from (z_t, y_t, a_t). Reward decoding dropped by design."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(LATENT_DIM + 2 * N, 128), nn.ReLU(),
                                 nn.Linear(128, N))
    def forward(self, z, y, a):
        return self.net(torch.cat([z, y, a], dim=-1))

class VariBADHook:
    def __init__(self, enc):
        self.enc = enc
    def reset(self):
        self.h = None
    def extra(self, obs):
        with torch.no_grad():
            mu, logvar, self.h = self.enc(
                torch.tensor(obs, dtype=torch.float32, device=device).view(1, 1, -1), self.h)
        mu = mu.view(-1).cpu().numpy()
        std = np.exp(0.5 * logvar.view(-1).cpu().numpy())
        return np.concatenate([mu, std])
    def observe(self, y, a, y_next):
        pass

def train_varibad(seed=303):
    np.random.seed(seed); torch.manual_seed(seed); random.seed(seed)
    enc = VariBADEncoder().to(device)
    dec = VariBADDecoder().to(device)
    policy = RecurrentAC(extra_dim=2 * LATENT_DIM).to(device)
    enc_opt = optim.AdamW(list(enc.parameters()) + list(dec.parameters()), lr=LR)
    pol_opt = optim.AdamW(policy.parameters(), lr=LR)
    hook = VariBADHook(enc)
    n_upd = TOTAL_EPISODES // EPISODES_PER_UPD
    ret_hist = []
    for upd in tqdm(range(n_upd), desc='VARIBAD'):
        eps = [rollout_episode(policy, OpponentTrainedP2(), hook)
               for _ in range(EPISODES_PER_UPD)]
        # ---- ELBO step (encoder + decoder) ----
        obs, ys, avecs = stack_batch(eps, ['obs', 'ys', 'avecs'])
        mu, logvar, _ = enc(obs)
        std = (0.5 * logvar).exp()
        z = mu + std * torch.randn_like(std)
        recon = dec(z, obs[..., :N], avecs)
        kl = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).sum(-1).mean()
        elbo = F.mse_loss(recon, ys) + VARIBAD_KL_W * kl
        enc_opt.zero_grad(); elbo.backward()
        torch.nn.utils.clip_grad_norm_(list(enc.parameters()) + list(dec.parameters()), 1.0)
        enc_opt.step()
        # ---- PPO step; beliefs fixed to their acting-time values ----
        extras = torch.tensor(np.stack([e['extras'] for e in eps]),
                              dtype=torch.float32, device=device)
        ppo_update(policy, pol_opt, eps, extras=extras)
        ret_hist.append(float(np.mean([e['rews'].mean() for e in eps])))
        if (upd + 1) % CKPT_EVERY == 0 or upd == n_upd - 1:
            torch.save({'policy': policy.state_dict(),
                        'encoder': enc.state_dict(),
                        'decoder': dec.state_dict(),
                        'config': {'RNN_H': RNN_H, 'LATENT_DIM': LATENT_DIM,
                                   'extra_dim': 2 * LATENT_DIM},
                        'reward_per_step': ret_hist},
                       os.path.join(SAVE_DIR, 'VARIBAD.pth'))
        if (upd + 1) % 25 == 0:
            print(f"VARIBAD upd {upd+1}/{n_upd}  reward/step (last 25 upd): "
                  f"{np.mean(ret_hist[-25:]):.3f}  elbo {elbo.item():.4f}")
    BUDGET['VARIBAD'] = n_upd * EPISODES_PER_UPD * max_steps
    return policy, enc, dec, ret_hist

VB_policy, VB_enc, VB_dec, VB_hist = train_varibad()

## PEARL: posterior sampling over a latent context

In [ ]:
class PEARLHook:
    def __init__(self, enc):
        self.enc = enc
    def reset(self):
        self.ctx = []
        self.z = np.random.randn(LATENT_DIM).astype(np.float32)   # prior sample
    def extra(self, obs):
        return self.z
    def observe(self, y, a, y_next):
        self.ctx.append(np.concatenate([y, a, y_next]).astype(np.float32))
        if len(self.ctx) % PEARL_RESAMPLE_K == 0:
            with torch.no_grad():
                mu, var = self.enc.posterior(
                    torch.tensor(np.stack(self.ctx), device=device))
            self.z = (mu + var.sqrt() * torch.randn_like(mu)).cpu().numpy()

def train_pearl(seed=404):
    np.random.seed(seed); torch.manual_seed(seed); random.seed(seed)
    enc = PEARLEncoder().to(device)
    policy = RecurrentAC(extra_dim=LATENT_DIM).to(device)
    opt = optim.AdamW(list(policy.parameters()) + list(enc.parameters()), lr=LR)
    hook = PEARLHook(enc)
    n_upd = TOTAL_EPISODES // EPISODES_PER_UPD
    ret_hist = []
    for upd in tqdm(range(n_upd), desc='PEARL'):
        eps = [rollout_episode(policy, OpponentTrainedP2(), hook)
               for _ in range(EPISODES_PER_UPD)]
        obs, acts, oldlp, adv, ret = stack_batch(eps, ['obs', 'acts', 'logp', 'adv', 'ret'])
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)
        extras_act = torch.tensor(np.stack([e['extras'] for e in eps]),
                                  dtype=torch.float32, device=device)
        trans = torch.tensor(np.stack(
            [np.concatenate([e['obs'][:, :N], e['avecs'], e['ys']], axis=-1) for e in eps]),
            dtype=torch.float32, device=device)          # (E, T, 3N)
        E = trans.shape[0]
        for _ in range(PPO_EPOCHS):
            # actor pass: acting-time latents, detached
            logits, _, _ = policy(obs, extras_act)
            dist = Categorical(logits=logits)
            lp = dist.log_prob(acts)
            ratio = (lp - oldlp).exp()
            pg = -torch.min(ratio * adv,
                            torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * adv).mean()
            ent = dist.entropy().mean()
            # critic pass: full-context posterior sample, gradient trains the encoder
            mus, vars_, zs = [], [], []
            for e_i in range(E):
                mu, var = enc.posterior(trans[e_i])
                zs.append(mu + var.sqrt() * torch.randn_like(mu))
                mus.append(mu); vars_.append(var)
            z_seq = torch.stack(zs).unsqueeze(1).expand(-1, max_steps, -1)
            _, values, _ = policy(obs, z_seq)
            vl = F.mse_loss(values, ret)
            mu_s = torch.stack(mus); var_s = torch.stack(vars_)
            kl = 0.5 * (var_s + mu_s.pow(2) - 1.0 - var_s.log()).sum(-1).mean()
            loss = pg + VF_COEF * vl - ENT_COEF * ent + PEARL_KL_W * kl
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(policy.parameters()) + list(enc.parameters()), 1.0)
            opt.step()
        ret_hist.append(float(np.mean([e['rews'].mean() for e in eps])))
        if (upd + 1) % CKPT_EVERY == 0 or upd == n_upd - 1:
            torch.save({'policy': policy.state_dict(),
                        'encoder': enc.state_dict(),
                        'config': {'RNN_H': RNN_H, 'LATENT_DIM': LATENT_DIM,
                                   'extra_dim': LATENT_DIM,
                                   'resample_k': PEARL_RESAMPLE_K},
                        'reward_per_step': ret_hist},
                       os.path.join(SAVE_DIR, 'PEARL.pth'))
        if (upd + 1) % 25 == 0:
            print(f"PEARL upd {upd+1}/{n_upd}  reward/step (last 25 upd): "
                  f"{np.mean(ret_hist[-25:]):.3f}")
    BUDGET['PEARL'] = n_upd * EPISODES_PER_UPD * max_steps
    return policy, enc, ret_hist

PEARL_policy, PEARL_enc, PEARL_hist = train_pearl()

## MMD: magnetic mirror descent in self-play

In [ ]:
class P2AC(nn.Module):
    def __init__(self):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(5 * N, 256), nn.ReLU(),
                                   nn.Linear(256, 256), nn.ReLU())
        self.pi = nn.Linear(256, n_u)
        self.v = nn.Linear(256, 1)
    def forward(self, inp, legal_mask):
        h = self.trunk(inp)
        logits = self.pi(h).masked_fill(~legal_mask, -1e9)
        return logits, self.v(h).squeeze(-1)

def rollout_selfplay(p1, p2):
    Theta, Weights, Reachable, x, legal = sample_config()
    tt = np.concatenate([Weights, Reachable]).astype(np.float32)
    legal_mask = torch.zeros(n_u, dtype=torch.bool, device=device)
    legal_mask[legal] = True
    m2 = np.full(N * 2, 0.5)
    h = None
    y_cur = np.zeros(N); a_prev = np.zeros(N)
    o1, a1, l1, v1, r1 = [], [], [], [], []
    i2, u2, l2, v2 = [], [], [], []
    for t in range(max_steps):
        obs = np.concatenate([y_cur, a_prev]).astype(np.float32)
        with torch.no_grad():
            logits, val, h = p1(torch.tensor(obs, device=device).view(1, 1, 2 * N), None, h)
        d1 = Categorical(logits=logits.view(-1))
        a = int(d1.sample())
        inp2 = np.concatenate([x, m2, tt]).astype(np.float32)
        with torch.no_grad():
            logits2, val2 = p2(torch.tensor(inp2, device=device).unsqueeze(0),
                               legal_mask.unsqueeze(0))
        d2 = Categorical(logits=logits2.view(-1))
        u = int(d2.sample())
        a_vec = actions[a]; u_vec = actions[u]
        nx = next_state(x, a_vec, u_vec, Theta)
        r = float(reward(nx, Theta))
        m2n = m2_update(m2, x, u_vec, nx)
        y = Observation(nx)
        o1.append(obs); a1.append(a)
        l1.append(float(d1.log_prob(torch.tensor(a, device=device))))
        v1.append(val.item()); r1.append(r)
        i2.append(inp2); u2.append(u)
        l2.append(float(d2.log_prob(torch.tensor(u, device=device))))
        v2.append(val2.item())
        x = nx; y_cur = y; a_prev = a_vec / B; m2 = m2n
    r1 = np.array(r1, dtype=np.float32)
    adv1, ret1 = gae(r1, np.array(v1, dtype=np.float32))
    adv2, ret2 = gae(-r1, np.array(v2, dtype=np.float32))
    ep1 = {'obs': np.stack(o1), 'acts': np.array(a1),
           'logp': np.array(l1, dtype=np.float32), 'adv': adv1, 'ret': ret1, 'rews': r1}
    ep2 = {'inp': np.stack(i2), 'acts': np.array(u2),
           'logp': np.array(l2, dtype=np.float32), 'adv': adv2, 'ret': ret2,
           'legal_mask': legal_mask}
    return ep1, ep2

def mmd_update_p1(p1, mag1, opt, eps):
    obs, acts, oldlp, adv, ret = stack_batch(eps, ['obs', 'acts', 'logp', 'adv', 'ret'])
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)
    for _ in range(PPO_EPOCHS):
        logits, values, _ = p1(obs)
        dist = Categorical(logits=logits)
        with torch.no_grad():
            mag_logits, _, _ = mag1(obs)
        kl_mag = kl_divergence(dist, Categorical(logits=mag_logits)).mean()
        lp = dist.log_prob(acts)
        ratio = (lp - oldlp).exp()
        pg = -torch.min(ratio * adv,
                        torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * adv).mean()
        vl = F.mse_loss(values, ret)
        loss = pg + VF_COEF * vl - ENT_COEF * dist.entropy().mean() + MMD_COEF * kl_mag
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(p1.parameters(), 1.0)
        opt.step()

def mmd_update_p2(p2, mag2, opt, eps2):
    inp = torch.tensor(np.stack([e['inp'] for e in eps2]),
                       dtype=torch.float32, device=device).view(-1, 5 * N)
    acts = torch.tensor(np.stack([e['acts'] for e in eps2]),
                        dtype=torch.long, device=device).view(-1)
    oldlp = torch.tensor(np.stack([e['logp'] for e in eps2]),
                         dtype=torch.float32, device=device).view(-1)
    adv = torch.tensor(np.stack([e['adv'] for e in eps2]),
                       dtype=torch.float32, device=device).view(-1)
    ret = torch.tensor(np.stack([e['ret'] for e in eps2]),
                       dtype=torch.float32, device=device).view(-1)
    mask = torch.stack([e['legal_mask'] for e in eps2]
                       ).unsqueeze(1).expand(-1, max_steps, -1).reshape(-1, n_u)
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)
    for _ in range(PPO_EPOCHS):
        logits, values = p2(inp, mask)
        dist = Categorical(logits=logits)
        with torch.no_grad():
            mag_logits, _ = mag2(inp, mask)
        kl_mag = kl_divergence(dist, Categorical(logits=mag_logits)).mean()
        lp = dist.log_prob(acts)
        ratio = (lp - oldlp).exp()
        pg = -torch.min(ratio * adv,
                        torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * adv).mean()
        vl = F.mse_loss(values, ret)
        loss = pg + VF_COEF * vl - ENT_COEF * dist.entropy().mean() + MMD_COEF * kl_mag
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(p2.parameters(), 1.0)
        opt.step()

def ema_(magnet, net):
    with torch.no_grad():
        for m, c in zip(magnet.parameters(), net.parameters()):
            m.mul_(1.0 - MAGNET_TAU).add_(MAGNET_TAU * c)

def train_mmd(seed=505):
    np.random.seed(seed); torch.manual_seed(seed); random.seed(seed)
    p1 = RecurrentAC(extra_dim=0).to(device)
    p2 = P2AC().to(device)
    mag1 = deepcopy(p1); mag2 = deepcopy(p2)
    mag1.gru.flatten_parameters()
    for m in list(mag1.parameters()) + list(mag2.parameters()):
        m.requires_grad_(False)
    opt1 = optim.AdamW(p1.parameters(), lr=LR)
    opt2 = optim.AdamW(p2.parameters(), lr=LR)
    n_upd = TOTAL_EPISODES // EPISODES_PER_UPD
    ret_hist = []
    for upd in tqdm(range(n_upd), desc='MMD'):
        pairs = [rollout_selfplay(p1, p2)
                 for _ in range(EPISODES_PER_UPD)]
        eps1 = [a for a, b in pairs]; eps2 = [b for a, b in pairs]
        mmd_update_p1(p1, mag1, opt1, eps1)
        mmd_update_p2(p2, mag2, opt2, eps2)
        ema_(mag1, p1); ema_(mag2, p2)
        ret_hist.append(float(np.mean([e['rews'].mean() for e in eps1])))
        if (upd + 1) % CKPT_EVERY == 0 or upd == n_upd - 1:
            torch.save({'p1': p1.state_dict(), 'p2': p2.state_dict(),
                        'p1_magnet': mag1.state_dict(), 'p2_magnet': mag2.state_dict(),
                        'config': {'RNN_H': RNN_H, 'extra_dim': 0,
                                   'MMD_COEF': MMD_COEF, 'MAGNET_TAU': MAGNET_TAU},
                        'reward_per_step': ret_hist},
                       os.path.join(SAVE_DIR, 'MMD.pth'))
        if (upd + 1) % 25 == 0:
            print(f"MMD upd {upd+1}/{n_upd}  P1 reward/step (last 25 upd): "
                  f"{np.mean(ret_hist[-25:]):.3f}")
    BUDGET['MMD'] = n_upd * EPISODES_PER_UPD * max_steps
    return p1, p2, ret_hist

MMD_p1, MMD_p2, MMD_hist = train_mmd()

## Interaction budgets

In [ ]:
print(f"{'method':8s} {'env steps':>10s}")
print(f"{'Ours':8s} {BUDGET_STEPS:10,d}   (main.ipynb)")
for k, v in BUDGET.items():
    print(f"{k:8s} {v:10,d}")